In [74]:
!pip install -q langchain-groq langchain-core requests

In [75]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
import json

In [76]:
#Tool create
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """ This function fetches the currency conversion factor between a given base currency and a target currency"""

  url = f'https://v6.exchangerate-api.com/v6/f6cb66c7c04a3a551d505bae/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_amount: int, conversion_rate: Annotated[float,InjectedToolArg]) -> float:
  """given a currency conversion rate this function calculates the target currency value from a given base currency value"""
  result = base_amount * conversion_rate
  return result

In [77]:
get_conversion_factor.invoke({"base_currency": 'USD',"target_currency":'INR'})['conversion_rate']

87.6968

In [78]:
convert.invoke({'base_amount':10, 'conversion_rate':85.16})

851.5999999999999

In [ ]:
# Tool binding
llm = ChatGroq(model = "llama3-8b-8192")

In [80]:
llm_with_tools = llm.bind_tools([convert,get_conversion_factor])

In [89]:
messages = [HumanMessage("What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd")]

In [90]:
ai_message = llm_with_tools.invoke(messages)

In [91]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '795jp83q3', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 1008, 'total_tokens': 1081, 'completion_time': 0.071761738, 'prompt_time': 0.112139503, 'queue_time': 0.002530736, 'total_time': 0.183901241}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_5b339000ab', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--fbd763bc-ed43-4a2d-915c-49f46a0b26aa-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '795jp83q3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1008, 'output_tokens': 73, 'total_tokens': 1081})

In [92]:
messages.append(ai_message)

In [93]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '795jp83q3',
  'type': 'tool_call'}]

In [94]:
for tool_call in ai_message.tool_calls:

  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)

  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [96]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '795jp83q3', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 1008, 'total_tokens': 1081, 'completion_time': 0.071761738, 'prompt_time': 0.112139503, 'queue_time': 0.002530736, 'total_time': 0.183901241}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_5b339000ab', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--fbd763bc-ed43-4a2d-915c-49f46a0b26aa-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '795jp83q3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1008, 'output_tokens

In [97]:
llm_with_tools.invoke(messages).content

''

In [98]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'convert',
   'description': 'given a currency conversion rate this function calculates the target currency value from a given base currency value',
   'parameters': {'properties': {'base_amount': {'type': 'integer'}},
    'required': ['base_amount'],
    'type': 'object'}}},
 {'type': 'function',
  'function': {'name': 'get_conversion_factor',
   'description': 'This function fetches the currency conversion factor between a given base currency and a target currency',
   'parameters': {'properties': {'base_currency': {'type': 'string'},
     'target_currency': {'type': 'string'}},
    'required': ['base_currency', 'target_currency'],
    'type': 'object'}}}]